Welcome to Lab 3 

We are going to set up the lab with a set of bash commands below. please go cell by cell 

these commands are going to assume you have ranthe setup script. if you haven't ran it yet then please do that now with the command below:


command assumes you are running a terminal inside the npu_lab folder. if doing this in vscode please just open a terminal as it will open it in the correct place

source steup.sh

if your device has a sudo pasword then run it in the terminal else run it via the cell below



In [ ]:
%%bash 
source setup.sh

Than you need to run the venv-setter script

source venv-setter.sh

same as before if you have a sudo password then you'll need to run it in the terminal 

In [ ]:
%%bash 
source venv-setter.sh

now select the kernel as the npu venv by using the top left menu in vscode or {insert method} in jupyter

Now we will get started with the content of the lab

**The creation of the models**

In [ ]:
import os
import torch
from torchvision import models

SAVE_DIR = "executorch_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

def to_executorch(pt_model: torch.nn.Module, example_inputs, out_path: str):
    pt_model.eval()

    exported = torch.export.export(pt_model, example_inputs)

    from executorch.exir import to_edge
    edge_mgr = to_edge(exported)

    et_mgr = edge_mgr.to_executorch()

    
    with open(out_path, "wb") as f:
        et_mgr.write_to_file(f)

    return out_path


model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
example = (torch.randn(1, 3, 224, 224),)

out_path = os.path.join(SAVE_DIR, "resnet18.pte")
to_executorch(model, example, out_path)
print(os.path.getsize(out_path), "bytes")
out_path


### Building a non-TOSA-compatible MobileNetV2

In this contrived example, we will construct a MobileNetV2-based model that intentionally breaks TOSA compatibility by inserting an LRN (Local Response Normalization) operation. This operation was proposed in the original AlexNet paper from 2012 and is not part of the TOSA specification. 

**What below cell does**
- Creates a backbone `MobileNetV2` 
- Inserts `tf.nn.local_response_normalization` (`LRN`) that is **not supported by TOSA**.
- Adds GAP + 1000-unit Dense “logits” layer and copies weights from a reference MobileNetV2 with top to preserve ImageNet logits.
- Exports a SavedModel and converts it to `.pte`.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights


SAVE_DIR = "executorch_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

class MobileNetV2_LRN_Break(nn.Module):
    """
    PyTorch equivalent of:
      MobileNetV2(include_top=False) -> LRN -> GAP -> Dense(1000)
    with Dense weights copied from pretrained MobileNetV2(include_top=True).
    """
    def __init__(self, backbone_weights="imagenet"):
        super().__init__()

        # Reference model WITH top to copy 1k linear weights
        if backbone_weights == "imagenet":
            ref = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
            feat = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
        else:
            ref = mobilenet_v2(weights=None)
            feat = mobilenet_v2(weights=None)

        # Backbone without top: torchvision splits into features + classifier
        self.features = feat.features  # outputs [N, 1280, 7, 7] for 224x224

        # LRN: TF depth_radius=2 => PyTorch LocalResponseNorm size=2*2+1=5
        # TF params: bias=1.0, alpha=1e-4, beta=0.75
        self.lrn = nn.LocalResponseNorm(size=5, alpha=1e-4, beta=0.75, k=1.0)

        # GAP + logits
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.logits = nn.Linear(1280, 1000)

        # Copy pretrained classifier weights (the Linear inside ref.classifier)
        # torchvision MobileNetV2 classifier is: Sequential(Dropout, Linear)
        ref_linear = None
        for m in ref.classifier.modules():
            if isinstance(m, nn.Linear) and m.out_features == 1000:
                ref_linear = m
                break
        if ref_linear is None:
            raise RuntimeError("Could not find 1000-way Linear layer in reference MobileNetV2.")

        with torch.no_grad():
            self.logits.weight.copy_(ref_linear.weight)
            self.logits.bias.copy_(ref_linear.bias)

    def forward(self, x):
        x = self.features(x)      # [N, C, H, W]
        x = self.lrn(x)           # LRN "break"
        x = self.pool(x)          # [N, C, 1, 1]
        x = torch.flatten(x, 1)   # [N, C]
        x = self.logits(x)        # [N, 1000] (logits)
        return x

# -------------------------
# ExecuTorch export 
# -------------------------
def to_executorch(pt_model: torch.nn.Module, example_inputs, out_path: str):
    pt_model.eval()
    exported = torch.export.export(pt_model, example_inputs)

    from executorch.exir import to_edge
    edge_mgr = to_edge(exported)
    et_mgr = edge_mgr.to_executorch()

    with open(out_path, "wb") as f:
        et_mgr.write_to_file(f)

    return out_path

# Build model (equivalent to mobilenetv2_with_lrn_break())
m_break = MobileNetV2_LRN_Break(backbone_weights="imagenet")

# Export to ExecuTorch .pte 
example = (torch.randn(1, 3, 224, 224),)
out_path = os.path.join(SAVE_DIR, "mnetv2_lrn_break.pte")
to_executorch(m_break, example, out_path)

out_path

# -------------------------
# ADDENDUM: Ethos-U quantize + lower to produce an FVP-ready .pte
# (Corstone-320 / Ethos-U85)
# -------------------------
import copy
import torch

def to_executorch_ethosu_fvp(
    pt_model: torch.nn.Module,
    example_inputs,
    out_path: str,
    *,
    target: str = "ethos-u85-128",
    system_config: str = "Ethos_U85_SYS_DRAM_Mid",
    memory_mode: str = "Shared_Sram",
    extra_flags: str | None = None,
):
    """
    Create a .pte suitable for the Corstone-320 FVP Ethos-U flow:
      - PT2E post-training quantization (int8)
      - Partition + lower to Ethos-U backend (TOSA + Vela compile)
      - Serialize .pte

    Notes:
    """
    pt_model.eval()

    # Ethos-U backend components
    from executorch.backends.arm.ethosu import EthosUPartitioner
    from executorch.backends.arm.quantizer.arm_quantizer import (
        EthosUQuantizer,
        get_symmetric_quantization_config,
    )
    from executorch.exir import (
        EdgeCompileConfig,
        ExecutorchBackendConfig,
        to_edge_transform_and_lower,
    )
    from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e

    # ---- Compile spec ----
    from executorch.backends.arm.ethosu import EthosUCompileSpec  # type: ignore

    compile_spec = EthosUCompileSpec(
            target=target,
            system_config=system_config,
            memory_mode=memory_mode,
            extra_flags=extra_flags or "",
        )

    # ---- PT2E post-training quantization (EthosUQuantizer) ----
    graph_module = torch.export.export(pt_model, example_inputs).module()

    quantizer = EthosUQuantizer(compile_spec)
    qconfig = get_symmetric_quantization_config(is_per_channel=False)
    quantizer.set_global(qconfig)

    graph_module = prepare_pt2e(graph_module, quantizer)
    graph_module(*example_inputs)  # calibration run (simple representative pass)
    graph_module = convert_pt2e(graph_module)

    quant_exported_program = torch.export.export(graph_module, example_inputs)

    # ---- Lower/partition to Ethos-U and write .pte ----
    edge_pm = to_edge_transform_and_lower(
        quant_exported_program,
        partitioner=[EthosUPartitioner(compile_spec)],
        compile_config=EdgeCompileConfig(_check_ir_validity=False),
    ).to_executorch(
        config=ExecutorchBackendConfig(extract_delegate_segments=False)
    )

    with open(out_path, "wb") as f:
        edge_pm.write_to_file(f)

    return out_path


# -------------------------
# Use it for your MobileNetV2_LRN_Break
# -------------------------
example = (torch.randn(1, 3, 224, 224),)

# LRN is commonly the thing that prevents a "fully integer" Ethos-U export.
# For the FVP/Ethos-U deploy build, bypass it.
m_fvp = copy.deepcopy(m_break)
m_fvp.lrn = torch.nn.Identity()

out_path = os.path.join(SAVE_DIR, "mnetv2_fvp_ethosu85_128.pte")
to_executorch_ethosu_fvp(
    m_fvp,
    example,
    out_path,
    target="ethos-u85-128",                 # must match your FVP num_macs
    system_config="Ethos_U85_SYS_DRAM_Mid", # common Corstone-320 DRAM config
    memory_mode="Shared_Sram",
)

print("Wrote:", out_path, "bytes:", os.path.getsize(out_path))


Add some words about lowering via the edge and transform function etc please

Lower to tosa of our above resnet model

## The Tensor Operator Set Architecture (TOSA)

### What is TOSA?
The **Tensor Operator Set Architecture (TOSA)** is a minimal and stable set of tensor-level operators designed to **standardize machine learning model execution across hardware platforms**.  
TOSA is framework-agnostic and includes precise functional and numerical definitions to ensure consistent results across CPUs, GPUs, and NPUs.

### Why Do We Need TOSA?
Modern ML frameworks such as [LiteRT](https://ai.google.dev/edge/litert), [Executorch](https://github.com/pytorch/executorch), and [JAX](https://github.com/jax-ml/jax) define hundreds (sometimes thousands) of distinct operators.  This diversity creates challenges when deploying models across platforms or converting them between frameworks (e.g., PyTorch → TensorFlow conversion is currently non-trivial due to different tensor layout conventions). 

**TOSA solves this problem** by providing a common set of standard operators that act as a stable intermediate representation (IR), a kind of “middle language” between source code and machine instructions that can be further compiled to a diverse range of backend targets. Subtle variations in the way operators are implemented in different frameworks, such as `CONV2D` for [2D convolutions](https://www.mlplatform.org/tosa/tosa_spec_1_0_1.html#_conv2d) are specified in detail so that ML developers a model will perform will perform as expected across devices and frameworks with an acceptably low level of rounding errors. 
 

### Key Benefits of TOSA
- **Standardization**: A consistent, well-defined target for both ML frameworks and hardware vendors.  
- **Portability**: TOSA-compliant models run on any hardware that supports TOSA, with guaranteed numerical consistency.  
- **Future-Proofing**: Future novel operators not directly in TOSA can still be expressed using existing TOSA primitives, ensuring long-term flexibility.  


---

## Additional Tools and Models Used in This Lab

- **[MobileNetV2 Model](https://huggingface.co/docs/transformers/en/model_doc/mobilenet_v2)**:  
  A convolutional neural network (CNN) architecture introduced by Google (2018), designed for mobile and edge devices where compute and memory are limited.  

- **Model Explorer**:  
  An Arm-built visualization tool that provides an **intuitive, hierarchical view** of model graphs.  
  It organizes model operations into nested layers, allowing users to dynamically expand or collapse them for easier analysis.
  Repository: [Model Explorer](https://model-explorer.staging.tools.arm.com) 



Now we will look at lowering our above resnet model we made right at the start that doesn't break the rules of tosa, into its tosa buffer to be read from a folder 

In [ ]:
from pathlib import Path
from torch.export import export
from executorch.exir import to_edge_transform_and_lower, EdgeCompileConfig
from executorch.backends.arm.tosa import TosaSpecification
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.backends.arm.util._factory import create_partitioner

model = model.eval()
ep = export(model,example) 

out_dir = Path("Executorch_intermdiates")
out_dir.mkdir(parents= True, exist_ok= True)

# 2) Create a TOSA compile spec (choose FP or INT profile)
tosa_spec = TosaSpecification.create_from_string("TOSA-1.0+FP")
compile_spec = TosaCompileSpec(tosa_spec)

# 3) Tell the backend to dump intermediates (this is where the .tosa appears)
out_dir = Path("tosa_dump")
out_dir.mkdir(parents=True, exist_ok=True)
compile_spec.dump_intermediate_artifacts_to(str(out_dir))

# 4) Partition + lower 
partitioner = create_partitioner(compile_spec)
edge_mgr = to_edge_transform_and_lower(
    ep,
    partitioner=[partitioner],
    compile_config=EdgeCompileConfig(_check_ir_validity=False),
)

print("Done. Look for a .tosa file under:", out_dir)



Here is where we use the model explorer to visualise the model, please select the file in the tosa_dump folder and then select model.pte or the example resnet.pte in our demo folder. 

The model explorer will let you visualise the models operations and their order. its best usages are in identify how the operations, their ordering and the inner workings of the model are different between formats in this lab we will ask yu to look at the difference ebtween .pte or the base executorch program and the .tosa format which is the series of buffers inside a model 


Copy this into your terminal to run the model explorer without cluttering the notebook: 



model-explorer --extension=pte_adapter_model_explorer


you can use this to visualise your .pte model in the base folder and your .tosa file found in the the above mentioned folder

there is an extension for .vgf and this will come in later labs

And here is where we can execute a set of commands to use the FVP. 

In [ ]:
%%bash 

source Repo/executorch/examples/arm/arm-scratch/setup_path.sh

That is the command to add the FVP's to PATH

In [ ]:
%%bash


source Repo/executorch/examples/arm/run.sh \
--aot_arm_compiler_flags="--delegate --quantize --intermediates mv2_u85/ --debug --evaluate" \
--output=mv2_u85 \
--target=ethos-u85-128 \
--model_name=mv2

above is the command for running our other model we built "Mobile net v2" on the ethos u 85 - You will see a a series of commands and potentially strange looking prompts. just wait until you see some stats appear. 

Please note that this command does a variety of things such as calling the "ahead of time" or aot compiler with a variety of flags that tell you what will happen to the model. the one we show in this lab has been set up in how the fvp would want it but we are going to use the complete arm workflow. 

this program will delegate the model to fit the ethos u work flow, it will quantize it as well. our target is simply the board and the model_name specifies what model from the executorch repo we are going to analyse. 